# Primer Parcial - Machine Learning 1
## Preprocesamiento de datos

### Pregunta de investigación

¿Qué factores académicos disponibles después del primer parcial permiten estimar la probabilidad de que un estudiante de Métodos Numéricos alcance una firma final de 71 puntos o más, considerando su rendimiento actual, su firma previa, su carga académica y su rendimiento en el ciclo anterior?

### Definición de la cohorte 

Se utilizarán como cohorte objetivo los estudiantes de Métodos Numéricos del segundo ciclo de 2025.

Para cada estudiante se utilizará:

- Información del primer ciclo de 2025 como historial académico previo.
- Información del segundo ciclo de 2025 disponible hasta el primer parcial.
- La firma obtenida al finalizar el segundo ciclo de 2025 como resultado a predecir.

La variable objetivo será `exonera`, definida como:

- `1`: firma final mayor o igual a 71.
- `0`: firma final menor a 71.

El análisis se plantea desde el momento posterior al primer parcial, por lo que no se utilizarán como predictores variables correspondientes a evaluaciones o resultados posteriores.

## 1. Importación de librerías

Se importan las librerías necesarias para la carga, manipulación y exploración inicial de los datos.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Carga de los datos

Se cargan los archivos correspondientes al primer y segundo ciclo de 2025.

El primer ciclo será utilizado para construir el historial académico de cada estudiante, mientras que el segundo ciclo contendrá la cohorte objetivo del análisis.

In [2]:
ruta_2025_01 = "../data/rendimiento_año_2025_ciclo_1_anon.xlsx"
ruta_2025_02 = "../data/rendimiento_año_2025_ciclo_2_anon.xlsx"

df_2025_01 = pd.read_excel(ruta_2025_01)
df_2025_02 = pd.read_excel(ruta_2025_02)

print("Dimensiones 2025-01:", df_2025_01.shape)
print("Dimensiones 2025-02:", df_2025_02.shape)

Dimensiones 2025-01: (19407, 29)
Dimensiones 2025-02: (20420, 28)


## 3. Revisión inicial de la estructura

Antes de realizar transformaciones se revisan las columnas disponibles en ambos períodos, ya que existen algunas diferencias entre los archivos.

In [3]:
print("Columnas 2025-01:")
print(df_2025_01.columns.tolist())

print("\nColumnas 2025-02:")
print(df_2025_02.columns.tolist())

Columnas 2025-01:
['ALUMNO_ID', 'Cod.Asign', 'Asignatura', 'Cod.Car.Sec', 'Cod.Curso', 'Convocatoria', 'Anho', 'Semestre', 'Doc.Firma', 'Aprobado', 'Anho.Firma', 'Primer.Par', 'Segundo.Par', 'Tercer.Par', 'TPLab.', 'Lab.', 'Proy.', 'Pond.PP', 'Pond.SP', 'Pond.TPLab', 'Pond.Lab', 'Pond.Proy', 'Asis', 'Requisito', 'Firma', 'Primer.Rec', 'Segundo.Rec', 'Nota.Final', 'FirmaCalculada']

Columnas 2025-02:
['ALUMNO_ID', 'Cod.Asign', 'Asignatura', 'Cod.Car.Sec', 'Cod.Curso', 'Convocatoria', 'Anho', 'Semestre', 'Doc.Firma', 'Aprobado', 'Anho.Firma', 'Primer.Par', 'Segundo.Par', 'Tercer.Par', 'TPLab.', 'Lab.', 'Proy.', 'Pond.PP', 'Pond.SP', 'Pond.TPLab', 'Pond.Lab', 'Pond.Proy', 'Asis', 'Requisito', 'Primer.Rec', 'Segundo.Rec', 'Nota.Final', 'Firma']


In [4]:
columnas_1 = set(df_2025_01.columns)
columnas_2 = set(df_2025_02.columns)

print("Solo en 2025-01:")
print(columnas_1 - columnas_2)

print("\nSolo en 2025-02:")
print(columnas_2 - columnas_1)

Solo en 2025-01:
{'FirmaCalculada'}

Solo en 2025-02:
set()


## 4. Selección de la asignatura Métodos Numéricos

Se filtran los registros correspondientes a la asignatura `Métodos Numéricos` en ambos períodos.

El segundo ciclo de 2025 representa la cohorte que será utilizada para el análisis, mientras que el primer ciclo permitirá obtener información previa de los mismos estudiantes.

In [5]:
mn_2025_01 = df_2025_01[df_2025_01["Asignatura"] == "METODOS NUMERICOS"].copy()

mn_2025_02 = df_2025_02[df_2025_02["Asignatura"] == "METODOS NUMERICOS"].copy()

print("Registros Métodos Numéricos 2025-01:", len(mn_2025_01))
print("Alumnos únicos 2025-01:", mn_2025_01["ALUMNO_ID"].nunique())

print("\nRegistros Métodos Numéricos 2025-02:", len(mn_2025_02))
print("Alumnos únicos 2025-02:", mn_2025_02["ALUMNO_ID"].nunique())

Registros Métodos Numéricos 2025-01: 760
Alumnos únicos 2025-01: 758

Registros Métodos Numéricos 2025-02: 260
Alumnos únicos 2025-02: 260


### Verificación de disponibilidad de historial

Se comprueba si los estudiantes de la cohorte objetivo también poseen registros
en el primer ciclo de 2025.

Esto permite determinar si es posible construir variables relacionadas con su
rendimiento académico previo.

In [6]:
alumnos_objetivo = set(mn_2025_02["ALUMNO_ID"])

alumnos_mn_2025_01 = set(mn_2025_01["ALUMNO_ID"])

alumnos_con_mn_previa = alumnos_objetivo.intersection(alumnos_mn_2025_01)

print("Alumnos objetivo:", len(alumnos_objetivo))
print("Con registro previo de Métodos Numéricos:", len(alumnos_con_mn_previa))
print("Sin registro previo:", len(alumnos_objetivo - alumnos_mn_2025_01))

Alumnos objetivo: 260
Con registro previo de Métodos Numéricos: 260
Sin registro previo: 0


## 5. Revisión de duplicados en Métodos Numéricos

El primer ciclo de 2025 contiene más registros que estudiantes únicos para la
asignatura. Por este motivo, antes de utilizar la información como historial se
identifican los estudiantes que poseen más de un registro.

In [7]:
duplicados_mn = (
    mn_2025_01["ALUMNO_ID"]
    .value_counts()
    .loc[lambda x: x > 1]
)

duplicados_mn

ALUMNO_ID
FIUNA_ALUMNO_010589    2
FIUNA_ALUMNO_011715    2
Name: count, dtype: int64

In [8]:
ids_duplicados = duplicados_mn.index

mn_2025_01[
    mn_2025_01["ALUMNO_ID"].isin(ids_duplicados)
].sort_values("ALUMNO_ID")

,ALUMNO_ID,Cod.Asign,Asignatura,Cod.Car.Sec,Cod.Curso,Convocatoria,Anho,Semestre,Doc.Firma,Aprobado,Anho.Firma,Primer.Par,Segundo.Par,Tercer.Par,TPLab.,Lab.,Proy.,Pond.PP,Pond.SP,Pond.TPLab,Pond.Lab,Pond.Proy,Asis,Requisito,Firma,Primer.Rec,Segundo.Rec,Nota.Final,FirmaCalculada
8026,FIUNA_ALUMNO_010589,13703,METODOS NUMERICOS,MCT-PLS13,4,1,2025,1,0,N,0,66,0,0,95,0,0,40,40,20,0,0,1,1,0,0,0,NaN,45
14824,FIUNA_ALUMNO_010589,23041,METODOS NUMERICOS,MCT-PLS23,4,1,2025,1,0,N,0,66,1,0,95,0,0,40,40,20,0,0,1,1,0,0,0,NaN,46
8027,FIUNA_ALUMNO_011715,13703,METODOS NUMERICOS,MCT-PLS13,4,1,2025,1,0,N,0,66,0,0,95,0,0,40,40,20,0,0,1,1,0,0,0,NaN,45
14825,FIUNA_ALUMNO_011715,23041,METODOS NUMERICOS,MCT-PLS23,4,1,2025,1,0,N,0,66,1,0,95,0,0,40,40,20,0,0,1,1,0,0,0,NaN,46


## 6. Distribución preliminar de la variable objetivo

Se analiza el puntaje de firma registrado en el segundo ciclo de 2025 para
determinar cuántos estudiantes alcanzaron el umbral de 71 puntos.

In [9]:
mn_2025_02["exonera"] = (mn_2025_02["Firma"] >= 71).astype(int)

mn_2025_02["exonera"].value_counts()

exonera
0    226
1     34
Name: count, dtype: int64

In [10]:
mn_2025_02["exonera"].value_counts(normalize=True) * 100

exonera
0    86.923077
1    13.076923
Name: proportion, dtype: float64

### Primer parcial del período objetivo

Se revisa la distribución del primer parcial de Métodos Numéricos en el segundo
ciclo de 2025, ya que esta variable representa el rendimiento actual disponible
en el momento definido para realizar la predicción.

In [11]:
mn_2025_02["Primer.Par"].describe()

count    260.000000
mean      27.669231
std       27.459310
min        0.000000
25%        0.000000
50%       25.000000
75%       47.000000
max      100.000000
Name: Primer.Par, dtype: float64

In [12]:
print("Estudiantes con Primer.Par > 0:",
      (mn_2025_02["Primer.Par"] > 0).sum())

print("Estudiantes con Primer.Par = 0:",
      (mn_2025_02["Primer.Par"] == 0).sum())

Estudiantes con Primer.Par > 0: 172
Estudiantes con Primer.Par = 0: 88


## 7. Construcción de la cohorte objetivo

La unidad de análisis será el estudiante.

Aunque la asignatura puede aparecer asociada a diferentes códigos o planes,
cada estudiante debe estar representado una sola vez en el conjunto final.

El segundo ciclo de 2025 se utilizará como período objetivo.

In [13]:
cohorte = mn_2025_02.copy()

print("Filas:", len(cohorte))
print("Alumnos únicos:", cohorte["ALUMNO_ID"].nunique())

Filas: 260
Alumnos únicos: 260


## 8. Construcción de la firma previa

Para cada estudiante se construye una variable `firma_previa` utilizando la
información de Métodos Numéricos del primer ciclo de 2025.

Debido a que un mismo estudiante puede aparecer asociado a más de un código de
asignatura y puede mejorar una firma obtenida previamente, se considera como
firma previa el mayor puntaje disponible entre `Firma` y `FirmaCalculada`.

Posteriormente, los registros se consolidan a nivel de estudiante.

### Consideración sobre los recuperatorios

En Métodos Numéricos, el primer recuperatorio puede reemplazar el peor puntaje
obtenido entre el primer y segundo parcial, permitiendo mejorar la firma final.

En el primer ciclo de 2025 existen registros con puntaje en `Primer.Rec`, por lo
que la variable `FirmaCalculada` se utiliza como parte de la construcción de la
firma previa, ya que refleja el resultado académico del período considerando
este mecanismo de recuperación.

En el segundo ciclo de 2025, utilizado como período objetivo, no se observan
puntajes distintos de cero en `Primer.Rec`, por lo que este mecanismo no afecta
directamente la variable objetivo en la cohorte analizada.

In [14]:
mn_2025_01["firma_previa_registro"] = mn_2025_01[["Firma", "FirmaCalculada"]].max(axis=1)

firma_previa = (mn_2025_01.groupby("ALUMNO_ID", as_index=False)["firma_previa_registro"]
    .max()
    .rename(columns={"firma_previa_registro": "firma_previa"})
)

firma_previa.head(10)

,ALUMNO_ID,firma_previa
0,FIUNA_ALUMNO_001346,48
1,FIUNA_ALUMNO_003114,48
2,FIUNA_ALUMNO_003887,35
3,FIUNA_ALUMNO_004333,35
4,FIUNA_ALUMNO_005151,27
5,FIUNA_ALUMNO_005166,51
6,FIUNA_ALUMNO_005188,0
7,FIUNA_ALUMNO_005496,50
8,FIUNA_ALUMNO_005561,53
9,FIUNA_ALUMNO_005607,15


In [15]:
firma_previa["firma_previa"].describe()

count    758.000000
mean      35.044855
std       22.413918
min        0.000000
25%       17.250000
50%       35.000000
75%       52.000000
max       92.000000
Name: firma_previa, dtype: float64

In [16]:
firma_previa_objetivo = firma_previa[firma_previa["ALUMNO_ID"].isin(cohorte["ALUMNO_ID"])]

firma_previa_objetivo["firma_previa"].describe()

count    260.000000
mean      42.446154
std       10.580131
min       20.000000
25%       34.000000
50%       41.500000
75%       52.000000
max       66.000000
Name: firma_previa, dtype: float64

## 9. Carga académica del ciclo anterior

Se calcula la cantidad de asignaturas diferentes cursadas por cada estudiante
durante el primer ciclo de 2025.

Esta variable busca representar la carga académica reciente del estudiante.
Se cuentan códigos de asignatura únicos para evitar duplicar una misma materia
por registros repetidos.

In [17]:
carga_anterior = (
    df_2025_01
    .groupby("ALUMNO_ID")["Cod.Asign"]
    .nunique()
    .reset_index(name="cantidad_materias_anterior")
)

carga_anterior.head(10)

,ALUMNO_ID,cantidad_materias_anterior
0,FIUNA_ALUMNO_000302,6
1,FIUNA_ALUMNO_000383,1
2,FIUNA_ALUMNO_000712,1
3,FIUNA_ALUMNO_000797,5
4,FIUNA_ALUMNO_000825,1
5,FIUNA_ALUMNO_000919,3
6,FIUNA_ALUMNO_000969,1
7,FIUNA_ALUMNO_001346,2
8,FIUNA_ALUMNO_001404,5
9,FIUNA_ALUMNO_001442,8


In [18]:
carga_anterior["cantidad_materias_anterior"].describe()

count    4000.000000
mean        4.851750
std         2.349181
min         1.000000
25%         3.000000
50%         5.000000
75%         6.000000
max        10.000000
Name: cantidad_materias_anterior, dtype: float64

In [19]:
df_2025_01["Aprobado"].value_counts(dropna=False)

Aprobado
S    11633
N     7774
Name: count, dtype: int64

In [20]:
df_2025_01["Aprobado_num"] = df_2025_01["Aprobado"].map({
    "S": 1,
    "N": 0
})

df_2025_01["Aprobado_num"].value_counts(dropna=False)

Aprobado_num
1    11633
0     7774
Name: count, dtype: int64

## 10. Rendimiento académico del ciclo anterior

Para representar el rendimiento reciente de cada estudiante se construyen tres
variables:

- cantidad de materias cursadas;
- cantidad de materias aprobadas;
- tasa de aprobación.

La tasa de aprobación se calcula como la proporción de asignaturas aprobadas
sobre el total de asignaturas cursadas en el ciclo anterior.

In [21]:
rendimiento_anterior = (
    df_2025_01
    .groupby("ALUMNO_ID")
    .agg(
        materias_aprobadas_anterior=("Aprobado_num", "sum"),
        cantidad_materias_anterior=("Cod.Asign", "nunique")
    )
    .reset_index()
)

rendimiento_anterior["tasa_aprobacion_anterior"] = (
    rendimiento_anterior["materias_aprobadas_anterior"] / rendimiento_anterior["cantidad_materias_anterior"]
)

rendimiento_anterior.head(10)

,ALUMNO_ID,materias_aprobadas_anterior,cantidad_materias_anterior,tasa_aprobacion_anterior
0,FIUNA_ALUMNO_000302,1,6,0.166667
1,FIUNA_ALUMNO_000383,0,1,0.000000
2,FIUNA_ALUMNO_000712,0,1,0.000000
3,FIUNA_ALUMNO_000797,2,5,0.400000
4,FIUNA_ALUMNO_000825,0,1,0.000000
5,FIUNA_ALUMNO_000919,0,3,0.000000
6,FIUNA_ALUMNO_000969,1,1,1.000000
7,FIUNA_ALUMNO_001346,2,2,1.000000
8,FIUNA_ALUMNO_001404,3,5,0.600000
9,FIUNA_ALUMNO_001442,7,8,0.875000


In [22]:
duplicados_asignatura = df_2025_01[
    df_2025_01.duplicated(
        subset=["ALUMNO_ID", "Cod.Asign"],
        keep=False
    )
][
    ["ALUMNO_ID", "Cod.Asign", "Asignatura", "Aprobado"]
].sort_values(["ALUMNO_ID", "Cod.Asign"])

duplicados_asignatura

,ALUMNO_ID,Cod.Asign,Asignatura,Aprobado


## 11. Carga académica del período actual

Se calcula la cantidad de asignaturas diferentes cursadas por cada estudiante
en el segundo ciclo de 2025.

Esta variable representa la carga académica concurrente al período en el que
se desea estimar la posibilidad de alcanzar una firma de 71 puntos.

In [23]:
carga_actual = (
    df_2025_02
    .groupby("ALUMNO_ID")["Cod.Asign"]
    .nunique()
    .reset_index(name="cantidad_materias_actual")
)

carga_actual.head()

,ALUMNO_ID,cantidad_materias_actual
0,FIUNA_ALUMNO_000302,4
1,FIUNA_ALUMNO_000383,1
2,FIUNA_ALUMNO_000660,1
3,FIUNA_ALUMNO_000676,1
4,FIUNA_ALUMNO_000797,3


## 12. Conversión del código de carrera

La columna `Cod.Car.Sec` contiene códigos asociados a carreras, planes e
intensificaciones.

Como el objetivo del análisis es estudiar diferencias entre carreras y no entre
planes o intensificaciones, estos códigos se agrupan en una única categoría
correspondiente a la carrera del estudiante.

In [24]:
career_code_mapping = {
    # Ingeniería Civil
    'CIV-PLS13': 'Ing. Civil',
    'CIV-PLS23': 'Ing. Civil',
    'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil',
    'INT9ORTERR': 'Ing. Civil',
    'INT9SANEHI': 'Ing. Civil',

    # Ingeniería Electromecánica
    'ELE-PLS13': 'Ing. Electromecánica',
    'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica',
    'INT9SDIGYT': 'Ing. Electromecánica',

    # Ingeniería Mecatrónica
    'MCT-PLS13': 'Ing. Mecatrónica',
    'MCT-PLS23': 'Ing. Mecatrónica',
    'MCT9-OPT': 'Ing. Mecatrónica',

    # Ingeniería Industrial
    'IND-PLS13': 'Ing. Industrial',
    'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial',
    'INT9-PROYT': 'Ing. Industrial',

    # Ingeniería Geográfica
    'CGF-PLS13': 'Ing. Geográfica',
    'CGF-PLS23': 'Ing. Geográfica',
    'INT9RNYMA': 'Ing. Geográfica',

    # Ingeniería Mecánica
    'MEC-PLS13': 'Ing. Mecánica',
    'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica',
    'MEC9-OPT': 'Ing. Mecánica',

    # Ingeniería Electrónica
    'ECA-PLS13': 'Ing. Electrónica',
    'ECA-PLS23': 'Ing. Electrónica',
    'ECA9-OPT': 'Ing. Electrónica'
}

In [25]:
mn_2025_02["Carrera"] = (
    mn_2025_02["Cod.Car.Sec"]
    .astype(str)
    .str.strip()
    .map(career_code_mapping)
)

In [26]:
mn_2025_02[
    mn_2025_02["Carrera"].isna()
]["Cod.Car.Sec"].value_counts()

Series([], Name: count, dtype: int64)

In [27]:
mn_2025_02["Carrera"].value_counts()

Carrera
Ing. Civil              108
Ing. Electromecánica     70
Ing. Geográfica          24
Ing. Industrial          20
Ing. Mecatrónica         18
Ing. Mecánica            12
Ing. Electrónica          8
Name: count, dtype: int64

### Consideración sobre la distribución por carrera

Se observa una distribución desigual entre carreras. Algunas carreras poseen
una cantidad reducida de estudiantes, por lo que las comparaciones entre grupos
deberán interpretarse con precaución.


## 13. Preparación de las variables actuales de Métodos Numéricos

Se seleccionan las variables correspondientes al segundo ciclo de 2025 que serán
utilizadas para representar la situación actual del estudiante en Métodos Numéricos.

Se conserva el puntaje del primer parcial porque corresponde a información disponible
en el momento definido para realizar la predicción.

También se conserva la firma registrada al finalizar el período, pero únicamente
para construir la variable objetivo. Esta variable no será utilizada como predictor,
ya que contiene información posterior al momento de predicción y produciría
data leakage.

In [28]:
actual = mn_2025_02[
    [
        "ALUMNO_ID",
        "Carrera",
        "Primer.Par",
        "Firma"
    ]
].copy()

actual = actual.rename(
    columns={
        "Carrera": "carrera",
        "Primer.Par": "primer_parcial",
        "Firma": "firma_actual"
    }
)

actual.head(10)

,ALUMNO_ID,carrera,primer_parcial,firma_actual
15100,FIUNA_ALUMNO_005925,Ing. Geográfica,0,18
15101,FIUNA_ALUMNO_007516,Ing. Geográfica,0,18
15102,FIUNA_ALUMNO_005561,Ing. Industrial,0,20
15103,FIUNA_ALUMNO_007838,Ing. Industrial,0,20
15104,FIUNA_ALUMNO_006031,Ing. Civil,0,20
15105,FIUNA_ALUMNO_007861,Ing. Civil,0,20
15106,FIUNA_ALUMNO_006420,Ing. Civil,67,66
15107,FIUNA_ALUMNO_007467,Ing. Civil,67,66
15108,FIUNA_ALUMNO_006043,Ing. Industrial,22,27
15109,FIUNA_ALUMNO_007865,Ing. Industrial,22,27


### Construcción de la variable objetivo

La variable `exonera` indica si el estudiante alcanzó una firma de 71 puntos o más
al finalizar el segundo ciclo de 2025:

- `1`: firma mayor o igual a 71.
- `0`: firma menor a 71.

La columna `firma_actual` se conserva únicamente para construir y verificar esta
variable objetivo, pero no se utilizará como predictor.

In [29]:
actual["exonera"] = (actual["firma_actual"] >= 71).astype(int)

actual["exonera"].value_counts()

exonera
0    226
1     34
Name: count, dtype: int64

## 14. Unión de las variables construidas

Se integran en una sola tabla las variables correspondientes al período actual
y al historial académico del ciclo anterior.

La tabla final tendrá una fila por estudiante de Métodos Numéricos del segundo
ciclo de 2025.

In [30]:
dataset = (
    actual
    .merge(
        firma_previa,
        on="ALUMNO_ID",
        how="left"
    )
    .merge(
        rendimiento_anterior,
        on="ALUMNO_ID",
        how="left"
    )
    .merge(
        carga_actual,
        on="ALUMNO_ID",
        how="left"
    )
)

dataset.head(10)

,ALUMNO_ID,carrera,primer_parcial,firma_actual,exonera,firma_previa,materias_aprobadas_anterior,cantidad_materias_anterior,tasa_aprobacion_anterior,cantidad_materias_actual
0,FIUNA_ALUMNO_005925,Ing. Geográfica,0,18,0,40,1,5,0.200000,7
1,FIUNA_ALUMNO_007516,Ing. Geográfica,0,18,0,40,1,5,0.200000,7
2,FIUNA_ALUMNO_005561,Ing. Industrial,0,20,0,53,3,7,0.428571,7
3,FIUNA_ALUMNO_007838,Ing. Industrial,0,20,0,53,3,7,0.428571,7
4,FIUNA_ALUMNO_006031,Ing. Civil,0,20,0,34,4,6,0.666667,6
5,FIUNA_ALUMNO_007861,Ing. Civil,0,20,0,34,4,6,0.666667,6
6,FIUNA_ALUMNO_006420,Ing. Civil,67,66,0,53,2,7,0.285714,7
7,FIUNA_ALUMNO_007467,Ing. Civil,67,66,0,53,2,7,0.285714,7
8,FIUNA_ALUMNO_006043,Ing. Industrial,22,27,0,32,3,6,0.500000,5
9,FIUNA_ALUMNO_007865,Ing. Industrial,22,27,0,32,3,6,0.500000,5


## 15. Verificación del dataset integrado

Se verifica que la cantidad de filas se mantenga igual a la cantidad de
estudiantes de la cohorte objetivo y que no se hayan generado duplicados
durante las uniones.

In [31]:
print("Dimensiones del dataset:", dataset.shape)
print("Cantidad de estudiantes únicos:", dataset["ALUMNO_ID"].nunique())
print("Cantidad total de filas:", len(dataset))

Dimensiones del dataset: (260, 10)
Cantidad de estudiantes únicos: 260
Cantidad total de filas: 260


In [32]:
dataset["ALUMNO_ID"].duplicated().sum()

np.int64(0)

## 16. Revisión de valores faltantes

Se analiza la presencia de valores faltantes en las variables construidas.

Esta revisión permite identificar si algún estudiante no posee información
suficiente del ciclo anterior o del período actual antes de continuar con el
modelado.

In [33]:
nulos = pd.DataFrame({
    "cantidad_nulos": dataset.isnull().sum(),
    "porcentaje_nulos": dataset.isnull().mean() * 100
})

nulos

,cantidad_nulos,porcentaje_nulos
ALUMNO_ID,0,0.0
carrera,0,0.0
primer_parcial,0,0.0
firma_actual,0,0.0
exonera,0,0.0
firma_previa,0,0.0
materias_aprobadas_anterior,0,0.0
cantidad_materias_anterior,0,0.0
tasa_aprobacion_anterior,0,0.0
cantidad_materias_actual,0,0.0


## 17. Revisión general de las variables

Se revisa la estructura del dataset final, los tipos de datos y las principales
estadísticas descriptivas antes de realizar transformaciones adicionales.

In [34]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ALUMNO_ID                    260 non-null    str    
 1   carrera                      260 non-null    str    
 2   primer_parcial               260 non-null    int64  
 3   firma_actual                 260 non-null    int64  
 4   exonera                      260 non-null    int64  
 5   firma_previa                 260 non-null    int64  
 6   materias_aprobadas_anterior  260 non-null    int64  
 7   cantidad_materias_anterior   260 non-null    int64  
 8   tasa_aprobacion_anterior     260 non-null    float64
 9   cantidad_materias_actual     260 non-null    int64  
dtypes: float64(1), int64(7), str(2)
memory usage: 20.4 KB


In [35]:
dataset.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ALUMNO_ID,260,260,FIUNA_ALUMNO_005925,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
carrera,260,7,Ing. Civil,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primer_parcial,260.0,NaN,NaN,NaN,27.669231,27.45931,0.0,0.0,25.0,47.0,100.0
firma_actual,260.0,NaN,NaN,NaN,37.915385,21.15982,0.0,20.0,31.5,56.0,95.0
exonera,260.0,NaN,NaN,NaN,0.130769,0.337798,0.0,0.0,0.0,0.0,1.0
firma_previa,260.0,NaN,NaN,NaN,42.446154,10.580131,20.0,34.0,41.5,52.0,66.0
materias_aprobadas_anterior,260.0,NaN,NaN,NaN,2.738462,1.569762,0.0,2.0,3.0,4.0,7.0
cantidad_materias_anterior,260.0,NaN,NaN,NaN,5.846154,1.369618,2.0,5.0,6.0,7.0,8.0
tasa_aprobacion_anterior,260.0,NaN,NaN,NaN,0.46544,0.226623,0.0,0.333333,0.5,0.625,0.875
cantidad_materias_actual,260.0,NaN,NaN,NaN,6.030769,1.394631,2.0,5.0,6.0,7.0,9.0


## 18. Distribución de la variable objetivo

Se analiza la distribución de la variable `exonera` para conocer la proporción
de estudiantes que alcanzaron una firma de 71 puntos o más.

In [36]:
dataset["exonera"].value_counts()

exonera
0    226
1     34
Name: count, dtype: int64

In [37]:
dataset["exonera"].value_counts(normalize=True) * 100

exonera
0    86.923077
1    13.076923
Name: proportion, dtype: float64

## 19. Revisión de la firma previa

Se verifica la distribución de la firma previa de los estudiantes de la cohorte.

Esto permite confirmar que ninguno de los estudiantes ya poseía una firma de
71 puntos o más antes del segundo ciclo de 2025.

In [38]:
dataset["firma_previa"].describe()

count    260.000000
mean      42.446154
std       10.580131
min       20.000000
25%       34.000000
50%       41.500000
75%       52.000000
max       66.000000
Name: firma_previa, dtype: float64

In [39]:
print(
    "Estudiantes con firma previa >= 71:",
    (dataset["firma_previa"] >= 71).sum()
)

Estudiantes con firma previa >= 71: 0


## 20. Revisión del primer parcial

El valor cero en `primer_parcial` se conserva como un valor válido.

Aunque en muchos casos podría representar una ausencia, el dataset no permite
distinguir con certeza entre ausencia y una calificación real de cero.

No se reemplaza por un valor faltante porque, independientemente de su causa,
representa una situación académica observable al momento de realizar la
predicción y podría aportar información al modelo.

Esta ambigüedad se considera una limitación del análisis.


In [40]:
dataset["primer_parcial"].describe()

count    260.000000
mean      27.669231
std       27.459310
min        0.000000
25%        0.000000
50%       25.000000
75%       47.000000
max      100.000000
Name: primer_parcial, dtype: float64

In [41]:
print(
    "Cantidad de estudiantes con primer_parcial = 0:",
    (dataset["primer_parcial"] == 0).sum()
)

Cantidad de estudiantes con primer_parcial = 0: 88


## 21. Revisión de las variables académicas construidas

Se revisan las variables relacionadas con la carga académica y el rendimiento
del ciclo anterior para detectar posibles valores inconsistentes.

In [42]:
dataset[
    [
        "cantidad_materias_actual",
        "cantidad_materias_anterior",
        "materias_aprobadas_anterior",
        "tasa_aprobacion_anterior"
    ]
].describe()

,cantidad_materias_actual,cantidad_materias_anterior,materias_aprobadas_anterior,tasa_aprobacion_anterior
count,260.000000,260.000000,260.000000,260.000000
mean,6.030769,5.846154,2.738462,0.465440
std,1.394631,1.369618,1.569762,0.226623
min,2.000000,2.000000,0.000000,0.000000
25%,5.000000,5.000000,2.000000,0.333333
50%,6.000000,6.000000,3.000000,0.500000
75%,7.000000,7.000000,4.000000,0.625000
max,9.000000,8.000000,7.000000,0.875000


In [43]:
print(
    "Tasa de aprobación menor que 0:",
    (dataset["tasa_aprobacion_anterior"] < 0).sum()
)

print(
    "Tasa de aprobación mayor que 1:",
    (dataset["tasa_aprobacion_anterior"] > 1).sum()
)

print(
    "Más aprobadas que materias cursadas:",
    (
        dataset["materias_aprobadas_anterior"]
        > dataset["cantidad_materias_anterior"]
    ).sum()
)

Tasa de aprobación menor que 0: 0
Tasa de aprobación mayor que 1: 0
Más aprobadas que materias cursadas: 0


### Consideración sobre variables relacionadas

Las variables `cantidad_materias_anterior`, `materias_aprobadas_anterior` y
`tasa_aprobacion_anterior` están relacionadas entre sí, ya que la tasa de
aprobación se calcula a partir de las dos primeras.

Por este motivo, inicialmente se conservan todas para el análisis exploratorio.
Posteriormente se evaluará su relación y posible redundancia antes del
entrenamiento del modelo.


## 22. Selección preliminar de variables

Para responder la pregunta de investigación se utilizarán como posibles
predictores únicamente variables disponibles hasta el momento posterior al
primer parcial.

Variables predictoras:

- `carrera`
- `primer_parcial`
- `firma_previa`
- `cantidad_materias_actual`
- `cantidad_materias_anterior`
- `materias_aprobadas_anterior`
- `tasa_aprobacion_anterior`

Variable objetivo:

- `exonera`

Las columnas `ALUMNO_ID` y `firma_actual` se conservarán únicamente para
identificación y validación, pero no serán utilizadas como predictores.

In [44]:
variables_modelo = [
    "carrera",
    "primer_parcial",
    "firma_previa",
    "cantidad_materias_actual",
    "cantidad_materias_anterior",
    "materias_aprobadas_anterior",
    "tasa_aprobacion_anterior",
    "exonera"
]

dataset_modelo = dataset[variables_modelo].copy()

dataset_modelo.head()

,carrera,primer_parcial,firma_previa,cantidad_materias_actual,cantidad_materias_anterior,materias_aprobadas_anterior,tasa_aprobacion_anterior,exonera
0,Ing. Geográfica,0,40,7,5,1,0.200000,0
1,Ing. Geográfica,0,40,7,5,1,0.200000,0
2,Ing. Industrial,0,53,7,7,3,0.428571,0
3,Ing. Industrial,0,53,7,7,3,0.428571,0
4,Ing. Civil,0,34,6,6,4,0.666667,0


## 23. Guardado de los datasets procesados

Se generan dos archivos:

- un dataset completo, que conserva variables auxiliares utilizadas para
  validación y análisis;
- un dataset destinado al modelado, que contiene únicamente los predictores
  seleccionados y la variable objetivo.

Las variables `ALUMNO_ID` y `firma_actual` no se incluyen en el dataset de
modelado para evitar utilizar identificadores o información posterior al
momento de predicción.


In [45]:
dataset.to_csv(
    "../data/dataset_metodos_numericos_completo.csv",
    index=False
)

dataset_modelo.to_csv(
    "../data/dataset_metodos_numericos_modelo.csv",
    index=False
)

print("Datasets guardados correctamente.")


Datasets guardados correctamente.
